In [1]:
import cv2
import numpy
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
import pandas as pd
pd.set_option('display.max_columns',None)

In [9]:
df=pd.read_csv("/content/drive/MyDrive/Car_detection_problem/train_solution_bounding_boxes.csv")

In [10]:
df.head()

,image,xmin,ymin,xmax,ymax
0,vid_4_1000.jpg,281.259045,187.035071,327.727931,223.225547
1,vid_4_10000.jpg,15.163531,187.035071,120.329957,236.430180
2,vid_4_10040.jpg,239.192475,176.764801,361.968162,236.430180
3,vid_4_10020.jpg,496.483358,172.363256,630.020260,231.539575
4,vid_4_10060.jpg,16.630970,186.546010,132.558611,238.386422


In [7]:
def generate_vid_id(img_name):
    l=img_name.strip().split('_')
    return l[0]+'_'+l[1]
df['video_id']=df['image'].apply(generate_vid_id)

In [8]:
df.head()

,image,xmin,ymin,xmax,ymax,video_id
0,vid_4_1000.jpg,281.259045,187.035071,327.727931,223.225547,vid_4
1,vid_4_10000.jpg,15.163531,187.035071,120.329957,236.430180,vid_4
2,vid_4_10040.jpg,239.192475,176.764801,361.968162,236.430180,vid_4
3,vid_4_10020.jpg,496.483358,172.363256,630.020260,231.539575,vid_4
4,vid_4_10060.jpg,16.630970,186.546010,132.558611,238.386422,vid_4


In [ ]:
df['video_id'].unique()

array(['vid_4'], dtype=object)

In [13]:
def generate_image_id(img):
  l=img.strip().split('_')
  img_id=l[2].split('.')[0]
  return int(img_id)
df['image_id']=df['image'].apply(generate_image_id)

In [14]:
df.head(20)

,image,xmin,ymin,xmax,ymax,image_id
0,vid_4_1000.jpg,281.259045,187.035071,327.727931,223.225547,1000
1,vid_4_10000.jpg,15.163531,187.035071,120.329957,236.430180,10000
2,vid_4_10040.jpg,239.192475,176.764801,361.968162,236.430180,10040
3,vid_4_10020.jpg,496.483358,172.363256,630.020260,231.539575,10020
4,vid_4_10060.jpg,16.630970,186.546010,132.558611,238.386422,10060
5,vid_4_10100.jpg,447.568741,160.625804,582.083936,232.517696,10100
6,vid_4_10120.jpg,168.755427,180.677284,304.738061,246.700450,10120
7,vid_4_10140.jpg,0.000000,188.991313,85.111433,249.145753,10140
8,vid_4_1020.jpg,202.506512,189.480373,239.192475,229.094273,1020
9,vid_4_1040.jpg,116.416787,189.969434,180.494935,229.094273,1040


In [17]:
df=df.sort_values(by='image_id').reset_index(drop=True)

In [28]:
grouped=df.groupby('image')[['xmin','ymin','xmax','ymax']].apply(lambda x:x.values.tolist())

In [29]:
grouped.iloc[1]

[[15.16353111, 187.0350708, 120.3299566, 236.4301802]]

In [30]:
unique_image=df['image'].unique()

In [18]:
from sklearn.model_selection import train_test_split

In [31]:
train_imgs,val_imgs=train_test_split(unique_image,test_size=0.2,shuffle=False)

In [34]:
import os
def convert_to_yolo(img_w,img_h,boxes,class_id=0):
    yolo_boxes=[]
    for x_min,y_min,x_max,y_max in boxes:
        x_center=(x_min+x_max)/2/img_w
        y_center=(y_min+y_max)/2/img_h
        width=(x_max-x_min)/img_w
        height=(y_max-y_min)/img_h
        yolo_boxes.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    return yolo_boxes

os.makedirs('/content/drive/MyDrive/Car_detection_problem/labels',exist_ok=True)
for img_name,boxes in grouped.items():
    img=cv2.imread(Path('/content/drive/MyDrive/Car_detection_problem/training_images')/img_name)
    h,w=img.shape[:2]
    yolo_lines=convert_to_yolo(w,h,boxes)

    label_file=img_name.replace('.jpg','.txt')
    with open(f"/content/drive/MyDrive/Car_detection_problem/labels/{label_file}",'w') as f:
        f.write('\n'.join(yolo_lines))

In [35]:
import shutil
for split,img_list in [('train',train_imgs),('val',val_imgs)]:
    os.makedirs(f'/content/drive/MyDrive/Car_detection_problem/dataset/images/{split}',exist_ok=True)
    os.makedirs(f'/content/drive/MyDrive/Car_detection_problem/dataset/labels/{split}',exist_ok=True)
    for img_name in img_list:
        shutil.copy(f'/content/drive/MyDrive/Car_detection_problem/training_images/{img_name}',f'/content/drive/MyDrive/Car_detection_problem/dataset/images/{split}/{img_name}')
        label_name=img_name.replace('.jpg','.txt')
        shutil.copy(f'/content/drive/MyDrive/Car_detection_problem/labels/{label_name}',f'/content/drive/MyDrive/Car_detection_problem/dataset/labels/{split}/{label_name}')

In [37]:
for root,dirs,files in os.walk('dataset'):
    print(f'Folder:{root}|Files:{len(files)}')